# **Pytorch Benchmark**

## Setup & GPU Info

In [1]:
# ============================================================
# PYTORCH BENCHMARK - Cell 1: Setup & GPU Info
# ============================================================
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import time
import json
import gc
import numpy as np
from collections import OrderedDict
import subprocess

print("=" * 70)
print("PYTORCH BENCHMARK SUITE (ROCm)")
print("=" * 70)

# Framework info
print(f"\nPyTorch version: {torch.__version__}")
print(f"CUDA/ROCm available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"Device count: {torch.cuda.device_count()}")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
    print(f"Device capability: {torch.cuda.get_device_capability(0)}")

    # Memory info
    total_mem = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"Total GPU Memory: {total_mem:.2f} GB")

    device = torch.device('cuda')
else:
    print("WARNING: No GPU detected! Running on CPU.")
    device = torch.device('cpu')

# Try to get ROCm info
try:
    result = subprocess.run(['rocm-smi', '--showid', '--showtemp', '--showuse'],
                          capture_output=True, text=True, timeout=10)
    print(f"\nROCm SMI Output:\n{result.stdout}")
except:
    print("\nCould not run rocm-smi")

# Results storage
results = {
    'framework': 'PyTorch',
    'version': torch.__version__,
    'device': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    'benchmarks': {}
}

# Utility functions
def sync_and_time():
    """Synchronize GPU and return current time"""
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    return time.perf_counter()

def warmup_gpu():
    """Warmup GPU with small operations"""
    if torch.cuda.is_available():
        for _ in range(10):
            a = torch.randn(256, 256, device=device)
            b = torch.randn(256, 256, device=device)
            c = torch.mm(a, b)
        torch.cuda.synchronize()
        del a, b, c
        torch.cuda.empty_cache()

def get_gpu_memory():
    """Get current GPU memory usage in MB"""
    if torch.cuda.is_available():
        return {
            'allocated': torch.cuda.memory_allocated(0) / (1024**2),
            'reserved': torch.cuda.memory_reserved(0) / (1024**2),
            'max_allocated': torch.cuda.max_memory_allocated(0) / (1024**2)
        }
    return {'allocated': 0, 'reserved': 0, 'max_allocated': 0}

def benchmark_fn(fn, num_runs=100, warmup_runs=20, label=""):
    """Benchmark a function with warmup and multiple runs"""
    # Warmup
    for _ in range(warmup_runs):
        fn()

    # Benchmark
    times = []
    for _ in range(num_runs):
        start = sync_and_time()
        fn()
        end = sync_and_time()
        times.append(end - start)

    times = np.array(times)
    result = {
        'mean_ms': float(np.mean(times) * 1000),
        'std_ms': float(np.std(times) * 1000),
        'min_ms': float(np.min(times) * 1000),
        'max_ms': float(np.max(times) * 1000),
        'median_ms': float(np.median(times) * 1000),
        'p95_ms': float(np.percentile(times, 95) * 1000),
        'p99_ms': float(np.percentile(times, 99) * 1000),
        'num_runs': num_runs
    }
    if label:
        print(f"  {label}: {result['mean_ms']:.3f} ± {result['std_ms']:.3f} ms "
              f"(min={result['min_ms']:.3f}, median={result['median_ms']:.3f})")
    return result

print("\n✅ Setup complete!")

PYTORCH BENCHMARK SUITE (ROCm)

PyTorch version: 2.10.0+rocm7.1
CUDA/ROCm available: True
Device count: 2
Device name: AMD Radeon RX 6800S
Device capability: (10, 3)
Total GPU Memory: 7.98 GB

ROCm SMI Output:


============================ ROCm System Management Interface ============================
=========================================== ID ===========================================
GPU[0]		: Device Name: 		AMD Radeon RX 6800S
GPU[0]		: Device ID: 		0x73ef
GPU[0]		: Device Rev: 		0xc0
GPU[0]		: Subsystem ID: 	0x1dcc
GPU[0]		: GUID: 		45168
GPU[1]		: Device Name: 		AMD Radeon 680M
GPU[1]		: Device ID: 		0x1681
GPU[1]		: Device Rev: 		0xc7
GPU[1]		: Subsystem ID: 	0x1dcc
GPU[1]		: GUID: 		56463
====================================== Temperature =======================================
GPU[0]		: Temperature (Sensor edge) (C): 55.0
GPU[0]		: Temperature (Sensor junction) (C): 55.0
GPU[0]		: Temperature (Sensor memory) (C): 52.0
GPU[1]		: Temperature (Sensor edge) (C): 57.0
=========

## Matrix Operations Bencmark

In [2]:
# ============================================================
# PYTORCH BENCHMARK - Cell 2: Matrix Operations
# ============================================================
print("=" * 70)
print("BENCHMARK 1: MATRIX OPERATIONS")
print("=" * 70)

results['benchmarks']['matmul'] = {}

# Matrix multiplication at various sizes
sizes = [256, 512, 1024, 2048, 4096, 8192]
dtypes_config = {
    'float32': torch.float32,
    'float16': torch.float16,
}

# Check if bfloat16 is supported
try:
    test = torch.randn(2, 2, dtype=torch.bfloat16, device=device)
    dtypes_config['bfloat16'] = torch.bfloat16
    del test
except:
    print("bfloat16 not supported on this device")

for dtype_name, dtype in dtypes_config.items():
    print(f"\n--- Matrix Multiplication ({dtype_name}) ---")
    results['benchmarks']['matmul'][dtype_name] = {}

    for size in sizes:
        try:
            warmup_gpu()
            a = torch.randn(size, size, dtype=dtype, device=device)
            b = torch.randn(size, size, dtype=dtype, device=device)

            def matmul_fn():
                c = torch.mm(a, b)
                return c

            # Adjust runs based on size
            num_runs = 200 if size <= 1024 else (100 if size <= 4096 else 50)
            r = benchmark_fn(matmul_fn, num_runs=num_runs, warmup_runs=20,
                           label=f"  Size {size}x{size}")

            # Calculate TFLOPS
            flops = 2 * size * size * size  # For matrix multiply
            tflops = flops / (r['mean_ms'] / 1000) / 1e12
            r['tflops'] = float(tflops)
            print(f"    → {tflops:.2f} TFLOPS")

            results['benchmarks']['matmul'][dtype_name][str(size)] = r

            del a, b
            torch.cuda.empty_cache()
        except RuntimeError as e:
            print(f"  Size {size}x{size}: SKIPPED (OOM or error: {e})")
            results['benchmarks']['matmul'][dtype_name][str(size)] = {'error': str(e)}

# Batched matrix multiplication
print(f"\n--- Batched Matrix Multiplication (float32) ---")
results['benchmarks']['batched_matmul'] = {}
batch_configs = [(32, 512, 512), (64, 256, 256), (128, 128, 128), (16, 1024, 1024)]

for batch, m, n in batch_configs:
    try:
        warmup_gpu()
        a = torch.randn(batch, m, n, dtype=torch.float32, device=device)
        b = torch.randn(batch, n, m, dtype=torch.float32, device=device)

        def batched_matmul_fn():
            return torch.bmm(a, b)

        r = benchmark_fn(batched_matmul_fn, num_runs=100, warmup_runs=20,
                       label=f"  Batch={batch}, Size={m}x{n}")

        flops = 2 * batch * m * n * m
        tflops = flops / (r['mean_ms'] / 1000) / 1e12
        r['tflops'] = float(tflops)
        print(f"    → {tflops:.2f} TFLOPS")

        key = f"b{batch}_{m}x{n}"
        results['benchmarks']['batched_matmul'][key] = r

        del a, b
        torch.cuda.empty_cache()
    except RuntimeError as e:
        print(f"  Config ({batch},{m},{n}): SKIPPED ({e})")

gc.collect()
torch.cuda.empty_cache()
print("\n✅ Matrix operations benchmark complete!")

BENCHMARK 1: MATRIX OPERATIONS

--- Matrix Multiplication (float32) ---
    Size 256x256: 0.040 ± 0.011 ms (min=0.038, median=0.039)
    → 0.84 TFLOPS
    Size 512x512: 0.092 ± 0.012 ms (min=0.089, median=0.090)
    → 2.91 TFLOPS
    Size 1024x1024: 0.273 ± 0.004 ms (min=0.270, median=0.272)
    → 7.87 TFLOPS
    Size 2048x2048: 2.352 ± 0.061 ms (min=2.172, median=2.344)
    → 7.30 TFLOPS
    Size 4096x4096: 17.888 ± 0.041 ms (min=17.787, median=17.891)
    → 7.68 TFLOPS
    Size 8192x8192: 142.894 ± 0.604 ms (min=141.874, median=143.088)
    → 7.69 TFLOPS

--- Matrix Multiplication (float16) ---
    Size 256x256: 0.033 ± 0.002 ms (min=0.032, median=0.033)
    → 1.01 TFLOPS
    Size 512x512: 0.063 ± 0.002 ms (min=0.062, median=0.063)
    → 4.24 TFLOPS
    Size 1024x1024: 0.184 ± 0.004 ms (min=0.179, median=0.184)
    → 11.66 TFLOPS
    Size 2048x2048: 1.182 ± 0.039 ms (min=1.120, median=1.169)
    → 14.54 TFLOPS
    Size 4096x4096: 10.033 ± 0.022 ms (min=9.990, median=10.030)
    → 13.

## Element-wise & Reduction Operations

In [3]:
# ============================================================
# PYTORCH BENCHMARK - Cell 3: Element-wise & Reduction Operations
# ============================================================
print("=" * 70)
print("BENCHMARK 2: ELEMENT-WISE & REDUCTION OPERATIONS")
print("=" * 70)

results['benchmarks']['elementwise'] = {}
results['benchmarks']['reductions'] = {}

tensor_sizes = [
    (1_000_000,),      # 1M
    (10_000_000,),     # 10M
    (50_000_000,),     # 50M
    (100_000_000,),    # 100M
]

# Element-wise operations
print("\n--- Element-wise Operations ---")
for shape in tensor_sizes:
    numel = shape[0]
    label = f"{numel/1e6:.0f}M"
    results['benchmarks']['elementwise'][label] = {}

    warmup_gpu()
    a = torch.randn(shape, device=device)
    b = torch.randn(shape, device=device)

    ops = {
        'add': lambda: torch.add(a, b),
        'mul': lambda: torch.mul(a, b),
        'exp': lambda: torch.exp(a),
        'sin': lambda: torch.sin(a),
        'sigmoid': lambda: torch.sigmoid(a),
        'tanh': lambda: torch.tanh(a),
        'relu': lambda: F.relu(a),
        'gelu': lambda: F.gelu(a),
        'silu': lambda: F.silu(a),
    }

    print(f"\n  Tensor size: {label} elements")
    for op_name, op_fn in ops.items():
        try:
            r = benchmark_fn(op_fn, num_runs=100, warmup_runs=20,
                           label=f"    {op_name}")
            # Bandwidth in GB/s (read input + write output)
            bytes_moved = numel * 4 * 2  # float32, read+write
            if op_name in ['add', 'mul']:
                bytes_moved = numel * 4 * 3  # 2 reads + 1 write
            bandwidth_gbs = bytes_moved / (r['mean_ms'] / 1000) / 1e9
            r['bandwidth_gbs'] = float(bandwidth_gbs)
            results['benchmarks']['elementwise'][label][op_name] = r
        except Exception as e:
            print(f"    {op_name}: ERROR - {e}")

    del a, b
    torch.cuda.empty_cache()

# Reduction operations
print(f"\n--- Reduction Operations ---")
for shape in tensor_sizes:
    numel = shape[0]
    label = f"{numel/1e6:.0f}M"
    results['benchmarks']['reductions'][label] = {}

    warmup_gpu()
    a = torch.randn(shape, device=device)

    ops = {
        'sum': lambda: torch.sum(a),
        'mean': lambda: torch.mean(a),
        'max': lambda: torch.max(a),
        'min': lambda: torch.min(a),
        'std': lambda: torch.std(a),
        'norm': lambda: torch.norm(a),
        'argmax': lambda: torch.argmax(a),
        'sort': lambda: torch.sort(a),
    }

    print(f"\n  Tensor size: {label} elements")
    for op_name, op_fn in ops.items():
        try:
            num_runs = 50 if op_name == 'sort' else 100
            r = benchmark_fn(op_fn, num_runs=num_runs, warmup_runs=10,
                           label=f"    {op_name}")
            results['benchmarks']['reductions'][label][op_name] = r
        except Exception as e:
            print(f"    {op_name}: ERROR - {e}")

    del a
    torch.cuda.empty_cache()

gc.collect()
torch.cuda.empty_cache()
print("\n✅ Element-wise & reduction benchmark complete!")

BENCHMARK 2: ELEMENT-WISE & REDUCTION OPERATIONS

--- Element-wise Operations ---

  Tensor size: 1M elements
      add: 0.039 ± 0.035 ms (min=0.031, median=0.033)
      mul: 0.033 ± 0.007 ms (min=0.031, median=0.031)
      exp: 0.029 ± 0.003 ms (min=0.027, median=0.028)
      sin: 0.031 ± 0.004 ms (min=0.026, median=0.028)
      sigmoid: 0.028 ± 0.002 ms (min=0.027, median=0.027)
      tanh: 0.035 ± 0.031 ms (min=0.026, median=0.030)
      relu: 0.029 ± 0.002 ms (min=0.027, median=0.027)
      gelu: 0.032 ± 0.022 ms (min=0.026, median=0.028)
      silu: 0.028 ± 0.002 ms (min=0.027, median=0.027)

  Tensor size: 10M elements
      add: 0.543 ± 0.006 ms (min=0.537, median=0.542)
      mul: 0.542 ± 0.006 ms (min=0.537, median=0.540)
      exp: 0.371 ± 0.003 ms (min=0.368, median=0.370)
      sin: 0.373 ± 0.010 ms (min=0.368, median=0.370)
      sigmoid: 0.372 ± 0.005 ms (min=0.368, median=0.370)
      tanh: 0.371 ± 0.002 ms (min=0.368, median=0.370)
      relu: 0.379 ± 0.006 ms (min=0.37

## Convolution Operations

In [4]:
# ============================================================
# PYTORCH BENCHMARK - Cell 4: Convolution Operations
# ============================================================
print("=" * 70)
print("BENCHMARK 3: CONVOLUTION OPERATIONS")
print("=" * 70)

results['benchmarks']['conv2d'] = {}

conv_configs = [
    # (batch, in_ch, out_ch, H, W, kernel, stride, padding, label)
    (32, 3, 64, 224, 224, 7, 2, 3, "ResNet-stem"),
    (32, 64, 64, 56, 56, 3, 1, 1, "ResNet-block1"),
    (32, 128, 128, 28, 28, 3, 1, 1, "ResNet-block2"),
    (32, 256, 256, 14, 14, 3, 1, 1, "ResNet-block3"),
    (32, 512, 512, 7, 7, 3, 1, 1, "ResNet-block4"),
    (64, 3, 32, 32, 32, 3, 1, 1, "CIFAR-style"),
    (16, 3, 64, 224, 224, 3, 1, 1, "VGG-style"),
    (1, 64, 64, 512, 512, 3, 1, 1, "HighRes-single"),
]

for batch, in_ch, out_ch, H, W, kernel, stride, padding, label in conv_configs:
    try:
        warmup_gpu()
        torch.cuda.reset_peak_memory_stats()

        x = torch.randn(batch, in_ch, H, W, device=device)
        conv = nn.Conv2d(in_ch, out_ch, kernel, stride=stride, padding=padding,
                        bias=False).to(device)

        # Forward only
        def conv_forward():
            return conv(x)

        print(f"\n  {label}: input=({batch},{in_ch},{H},{W}), "
              f"kernel={kernel}x{kernel}, out_ch={out_ch}")

        r_fwd = benchmark_fn(conv_forward, num_runs=100, warmup_runs=20,
                            label=f"    Forward")

        # Forward + backward
        def conv_fwd_bwd():
            out = conv(x)
            loss = out.sum()
            loss.backward()
            conv.zero_grad()

        r_fwd_bwd = benchmark_fn(conv_fwd_bwd, num_runs=50, warmup_runs=10,
                                label=f"    Fwd+Bwd")

        mem = get_gpu_memory()

        results['benchmarks']['conv2d'][label] = {
            'forward': r_fwd,
            'fwd_bwd': r_fwd_bwd,
            'peak_memory_mb': mem['max_allocated'],
            'config': {
                'batch': batch, 'in_ch': in_ch, 'out_ch': out_ch,
                'H': H, 'W': W, 'kernel': kernel, 'stride': stride
            }
        }

        del x, conv
        torch.cuda.empty_cache()

    except RuntimeError as e:
        print(f"  {label}: SKIPPED ({e})")
        results['benchmarks']['conv2d'][label] = {'error': str(e)}

gc.collect()
torch.cuda.empty_cache()
print("\n✅ Convolution benchmark complete!")

BENCHMARK 3: CONVOLUTION OPERATIONS

  ResNet-stem: input=(32,3,224,224), kernel=7x7, out_ch=64
      Forward: 1.150 ± 0.079 ms (min=1.098, median=1.129)
      Fwd+Bwd: 36.808 ± 0.130 ms (min=36.667, median=36.726)

  ResNet-block1: input=(32,64,56,56), kernel=3x3, out_ch=64
      Forward: 0.545 ± 0.014 ms (min=0.530, median=0.544)
      Fwd+Bwd: 2.909 ± 0.087 ms (min=2.752, median=2.924)

  ResNet-block2: input=(32,128,28,28), kernel=3x3, out_ch=128
      Forward: 0.588 ± 0.015 ms (min=0.565, median=0.588)
      Fwd+Bwd: 1.667 ± 0.067 ms (min=1.607, median=1.642)

  ResNet-block3: input=(32,256,14,14), kernel=3x3, out_ch=256
      Forward: 0.594 ± 0.024 ms (min=0.563, median=0.590)
      Fwd+Bwd: 1.237 ± 0.062 ms (min=1.196, median=1.228)

  ResNet-block4: input=(32,512,7,7), kernel=3x3, out_ch=512
      Forward: 0.710 ± 0.023 ms (min=0.670, median=0.712)
      Fwd+Bwd: 1.459 ± 0.034 ms (min=1.417, median=1.449)

  CIFAR-style: input=(64,3,32,32), kernel=3x3, out_ch=32
      Forward: 

## Cell 5: Common Layer Operations

In [5]:
# ============================================================
# PYTORCH BENCHMARK - Cell 5: Common Layer Operations
# ============================================================
print("=" * 70)
print("BENCHMARK 4: COMMON LAYER OPERATIONS")
print("=" * 70)

results['benchmarks']['layers'] = {}

warmup_gpu()

# --- Linear Layer ---
print("\n--- Linear Layers ---")
linear_configs = [
    (128, 768, 768, "BERT-hidden"),
    (128, 768, 3072, "BERT-FFN-up"),
    (128, 3072, 768, "BERT-FFN-down"),
    (32, 1024, 4096, "Large-FFN-up"),
    (32, 4096, 1024, "Large-FFN-down"),
    (1, 4096, 4096, "LLM-single"),
]

results['benchmarks']['layers']['linear'] = {}
for batch, in_feat, out_feat, label in linear_configs:
    try:
        warmup_gpu()
        x = torch.randn(batch, in_feat, device=device)
        layer = nn.Linear(in_feat, out_feat, bias=True).to(device)

        def linear_fwd():
            return layer(x)

        def linear_fwd_bwd():
            out = layer(x)
            loss = out.sum()
            loss.backward()
            layer.zero_grad()

        r_fwd = benchmark_fn(linear_fwd, num_runs=200, warmup_runs=30,
                            label=f"  {label} fwd")
        r_bwd = benchmark_fn(linear_fwd_bwd, num_runs=100, warmup_runs=20,
                            label=f"  {label} fwd+bwd")

        results['benchmarks']['layers']['linear'][label] = {
            'forward': r_fwd, 'fwd_bwd': r_bwd
        }
        del x, layer
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"  {label}: ERROR - {e}")

# --- BatchNorm ---
print("\n--- BatchNorm ---")
results['benchmarks']['layers']['batchnorm'] = {}
bn_configs = [
    (32, 64, 56, 56, "BN-early"),
    (32, 256, 14, 14, "BN-mid"),
    (32, 512, 7, 7, "BN-late"),
]

for batch, ch, H, W, label in bn_configs:
    try:
        warmup_gpu()
        x = torch.randn(batch, ch, H, W, device=device)
        bn = nn.BatchNorm2d(ch).to(device)

        def bn_fwd():
            return bn(x)

        r = benchmark_fn(bn_fwd, num_runs=200, warmup_runs=30, label=f"  {label}")
        results['benchmarks']['layers']['batchnorm'][label] = r
        del x, bn
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"  {label}: ERROR - {e}")

# --- LayerNorm ---
print("\n--- LayerNorm ---")
results['benchmarks']['layers']['layernorm'] = {}
ln_configs = [
    ((128, 128, 768), (768,), "BERT-LN"),
    ((32, 256, 1024), (1024,), "Large-LN"),
    ((1, 2048, 4096), (4096,), "LLM-LN"),
]

for shape, norm_shape, label in ln_configs:
    try:
        warmup_gpu()
        x = torch.randn(shape, device=device)
        ln = nn.LayerNorm(norm_shape).to(device)

        def ln_fwd():
            return ln(x)

        r = benchmark_fn(ln_fwd, num_runs=200, warmup_runs=30, label=f"  {label}")
        results['benchmarks']['layers']['layernorm'][label] = r
        del x, ln
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"  {label}: ERROR - {e}")

# --- Multi-Head Attention ---
print("\n--- Multi-Head Attention ---")
results['benchmarks']['layers']['attention'] = {}
attn_configs = [
    (32, 128, 768, 12, "BERT-attn"),
    (16, 256, 768, 12, "BERT-long-attn"),
    (8, 512, 1024, 16, "Large-attn"),
    (1, 1024, 1024, 16, "LLM-attn"),
    (1, 2048, 768, 12, "Long-seq-attn"),
]

for batch, seq_len, embed, heads, label in attn_configs:
    try:
        warmup_gpu()
        mha = nn.MultiheadAttention(embed, heads, batch_first=True).to(device)
        x = torch.randn(batch, seq_len, embed, device=device)

        def attn_fwd():
            return mha(x, x, x, need_weights=False)

        def attn_fwd_bwd():
            out, _ = mha(x, x, x, need_weights=False)
            loss = out.sum()
            loss.backward()
            mha.zero_grad()

        r_fwd = benchmark_fn(attn_fwd, num_runs=50, warmup_runs=10,
                            label=f"  {label} fwd")
        r_bwd = benchmark_fn(attn_fwd_bwd, num_runs=30, warmup_runs=10,
                            label=f"  {label} fwd+bwd")

        results['benchmarks']['layers']['attention'][label] = {
            'forward': r_fwd, 'fwd_bwd': r_bwd
        }
        del mha, x
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"  {label}: ERROR - {e}")

# --- Softmax ---
print("\n--- Softmax ---")
results['benchmarks']['layers']['softmax'] = {}
for size in [(128, 128, 768), (32, 256, 1024), (8, 512, 4096)]:
    try:
        warmup_gpu()
        x = torch.randn(size, device=device)
        label = f"{'x'.join(map(str, size))}"

        def softmax_fn():
            return F.softmax(x, dim=-1)

        r = benchmark_fn(softmax_fn, num_runs=200, warmup_runs=30, label=f"  {label}")
        results['benchmarks']['layers']['softmax'][label] = r
        del x
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"  {label}: ERROR - {e}")

gc.collect()
torch.cuda.empty_cache()
print("\n✅ Layer operations benchmark complete!")

BENCHMARK 4: COMMON LAYER OPERATIONS

--- Linear Layers ---
    BERT-hidden fwd: 0.092 ± 0.006 ms (min=0.086, median=0.090)
    BERT-hidden fwd+bwd: 0.236 ± 0.047 ms (min=0.186, median=0.229)
    BERT-FFN-up fwd: 0.151 ± 0.015 ms (min=0.143, median=0.148)
    BERT-FFN-up fwd+bwd: 0.302 ± 0.006 ms (min=0.297, median=0.301)
    BERT-FFN-down fwd: 0.288 ± 0.017 ms (min=0.277, median=0.283)
    BERT-FFN-down fwd+bwd: 0.486 ± 0.041 ms (min=0.466, median=0.481)
    Large-FFN-up fwd: 0.194 ± 0.008 ms (min=0.188, median=0.191)
    Large-FFN-up fwd+bwd: 0.298 ± 0.019 ms (min=0.284, median=0.292)
    Large-FFN-down fwd: 0.254 ± 0.013 ms (min=0.239, median=0.249)
    Large-FFN-down fwd+bwd: 0.362 ± 0.012 ms (min=0.343, median=0.358)
    LLM-single fwd: 0.468 ± 0.010 ms (min=0.453, median=0.466)
    LLM-single fwd+bwd: 0.930 ± 0.054 ms (min=0.860, median=0.923)

--- BatchNorm ---
    BN-early: 0.278 ± 0.020 ms (min=0.266, median=0.271)
    BN-mid: 0.085 ± 0.012 ms (min=0.077, median=0.079)
    BN-

## CNN Model Training

In [6]:
# ============================================================
# PYTORCH BENCHMARK - Cell 6: CNN Model Training
# ============================================================
print("=" * 70)
print("BENCHMARK 5: CNN MODEL TRAINING (ResNet-like)")
print("=" * 70)

results['benchmarks']['cnn_training'] = {}

class BasicBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, stride=1, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_ch)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_ch != out_ch:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_ch)
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)

class SimpleResNet(nn.Module):
    def __init__(self, num_classes=1000):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 7, stride=2, padding=3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.pool = nn.MaxPool2d(3, stride=2, padding=1)

        self.layer1 = self._make_layer(64, 64, 2, stride=1)
        self.layer2 = self._make_layer(64, 128, 2, stride=2)
        self.layer3 = self._make_layer(128, 256, 2, stride=2)
        self.layer4 = self._make_layer(256, 512, 2, stride=2)

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, in_ch, out_ch, blocks, stride):
        layers = [BasicBlock(in_ch, out_ch, stride)]
        for _ in range(1, blocks):
            layers.append(BasicBlock(out_ch, out_ch, 1))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = self.pool(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.fc(x)
        return x

# Count parameters
model_test = SimpleResNet(1000)
total_params = sum(p.numel() for p in model_test.parameters())
trainable_params = sum(p.numel() for p in model_test.parameters() if p.requires_grad)
print(f"\nModel: SimpleResNet-18-like")
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
del model_test

# Training benchmark
batch_sizes = [16, 32, 64]
input_size = 224
num_classes = 1000

for batch_size in batch_sizes:
    label = f"batch_{batch_size}"
    print(f"\n--- Batch Size: {batch_size} ---")

    try:
        warmup_gpu()
        torch.cuda.reset_peak_memory_stats()

        model = SimpleResNet(num_classes).to(device)
        optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=1e-4)
        criterion = nn.CrossEntropyLoss()

        x = torch.randn(batch_size, 3, input_size, input_size, device=device)
        y = torch.randint(0, num_classes, (batch_size,), device=device)

        # Forward only (inference)
        model.eval()
        def inference_fn():
            with torch.no_grad():
                return model(x)

        r_infer = benchmark_fn(inference_fn, num_runs=50, warmup_runs=10,
                              label=f"  Inference")
        throughput_infer = batch_size / (r_infer['mean_ms'] / 1000)
        r_infer['throughput_imgs_per_sec'] = float(throughput_infer)
        print(f"    → {throughput_infer:.1f} images/sec")

        # Training step
        model.train()
        def train_step():
            optimizer.zero_grad()
            output = model(x)
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()

        r_train = benchmark_fn(train_step, num_runs=50, warmup_runs=10,
                              label=f"  Training step")
        throughput_train = batch_size / (r_train['mean_ms'] / 1000)
        r_train['throughput_imgs_per_sec'] = float(throughput_train)
        print(f"    → {throughput_train:.1f} images/sec")

        mem = get_gpu_memory()
        print(f"    Peak memory: {mem['max_allocated']:.1f} MB")

        results['benchmarks']['cnn_training'][label] = {
            'inference': r_infer,
            'training': r_train,
            'peak_memory_mb': mem['max_allocated']
        }

        del model, optimizer, criterion, x, y
        torch.cuda.empty_cache()
        gc.collect()

    except RuntimeError as e:
        print(f"  SKIPPED: {e}")
        results['benchmarks']['cnn_training'][label] = {'error': str(e)}

# --- Mixed Precision Training ---
print(f"\n--- Mixed Precision Training (AMP) ---")
try:
    from torch.cuda.amp import autocast, GradScaler

    batch_size = 32
    warmup_gpu()
    torch.cuda.reset_peak_memory_stats()

    model = SimpleResNet(num_classes).to(device)
    optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    criterion = nn.CrossEntropyLoss()
    scaler = GradScaler()

    x = torch.randn(batch_size, 3, input_size, input_size, device=device)
    y = torch.randint(0, num_classes, (batch_size,), device=device)

    model.train()
    def amp_train_step():
        optimizer.zero_grad()
        with autocast():
            output = model(x)
            loss = criterion(output, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

    r_amp = benchmark_fn(amp_train_step, num_runs=50, warmup_runs=10,
                        label=f"  AMP Training (batch={batch_size})")
    throughput_amp = batch_size / (r_amp['mean_ms'] / 1000)
    r_amp['throughput_imgs_per_sec'] = float(throughput_amp)
    print(f"    → {throughput_amp:.1f} images/sec")

    mem = get_gpu_memory()
    print(f"    Peak memory: {mem['max_allocated']:.1f} MB")

    results['benchmarks']['cnn_training']['amp_batch_32'] = {
        'training': r_amp,
        'peak_memory_mb': mem['max_allocated']
    }

    del model, optimizer, criterion, scaler, x, y
    torch.cuda.empty_cache()
    gc.collect()

except Exception as e:
    print(f"  AMP not available or failed: {e}")

print("\n✅ CNN training benchmark complete!")

BENCHMARK 5: CNN MODEL TRAINING (ResNet-like)

Model: SimpleResNet-18-like
Total parameters: 11,689,512
Trainable parameters: 11,689,512

--- Batch Size: 16 ---
    Inference: 8.206 ± 0.120 ms (min=7.812, median=8.229)
    → 1949.7 images/sec
    Training step: 49.945 ± 0.057 ms (min=49.799, median=49.941)
    → 320.4 images/sec
    Peak memory: 525.5 MB

--- Batch Size: 32 ---
    Inference: 17.162 ± 0.112 ms (min=16.965, median=17.141)
    → 1864.6 images/sec
    Training step: 98.347 ± 2.263 ms (min=97.424, median=97.754)
    → 325.4 images/sec
    Peak memory: 857.1 MB

--- Batch Size: 64 ---
    Inference: 35.275 ± 0.082 ms (min=35.170, median=35.253)
    → 1814.3 images/sec
    Training step: 196.731 ± 0.553 ms (min=196.155, median=196.692)
    → 325.3 images/sec
    Peak memory: 1531.9 MB

--- Mixed Precision Training (AMP) ---


/tmp/ipykernel_129244/1196561109.py:149: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_129244/1196561109.py:157: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


    AMP Training (batch=32): 64.335 ± 0.139 ms (min=64.085, median=64.343)
    → 497.4 images/sec
    Peak memory: 576.0 MB

✅ CNN training benchmark complete!


## Transformer Model Training

In [7]:
# ============================================================
# PYTORCH BENCHMARK - Cell 7: Transformer Model Training
# ============================================================
print("=" * 70)
print("BENCHMARK 6: TRANSFORMER MODEL TRAINING")
print("=" * 70)

results['benchmarks']['transformer_training'] = {}

class TransformerModel(nn.Module):
    def __init__(self, vocab_size=30000, d_model=512, nhead=8,
                 num_layers=6, dim_ff=2048, max_seq_len=512, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.pos_encoding = nn.Parameter(torch.randn(1, max_seq_len, d_model) * 0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_ff,
            batch_first=True, dropout=0.1
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.classifier = nn.Linear(d_model, num_classes)
        self.d_model = d_model

    def forward(self, x):
        x = self.embedding(x) * (self.d_model ** 0.5)
        seq_len = x.size(1)
        x = x + self.pos_encoding[:, :seq_len, :]
        x = self.encoder(x)
        x = x.mean(dim=1)
        return self.classifier(x)

# Transformer configs
transformer_configs = [
    # (d_model, nhead, layers, ff_dim, seq_len, batch, label)
    (256, 4, 4, 1024, 128, 64, "Small-Transformer"),
    (512, 8, 6, 2048, 128, 32, "BERT-base-like"),
    (512, 8, 6, 2048, 256, 16, "BERT-long-seq"),
    (768, 12, 6, 3072, 128, 16, "BERT-large-like"),
]

for d_model, nhead, num_layers, dim_ff, seq_len, batch_size, label in transformer_configs:
    print(f"\n--- {label} ---")
    print(f"  Config: d_model={d_model}, heads={nhead}, layers={num_layers}, "
          f"ff={dim_ff}, seq={seq_len}, batch={batch_size}")

    try:
        warmup_gpu()
        torch.cuda.reset_peak_memory_stats()

        model = TransformerModel(
            d_model=d_model, nhead=nhead, num_layers=num_layers,
            dim_ff=dim_ff, max_seq_len=seq_len
        ).to(device)

        param_count = sum(p.numel() for p in model.parameters())
        print(f"  Parameters: {param_count:,}")

        optimizer = optim.Adam(model.parameters(), lr=1e-4)
        criterion = nn.CrossEntropyLoss()

        x = torch.randint(0, 30000, (batch_size, seq_len), device=device)
        y = torch.randint(0, 2, (batch_size,), device=device)

        # Inference
        model.eval()
        def infer_fn():
            with torch.no_grad():
                return model(x)

        r_infer = benchmark_fn(infer_fn, num_runs=30, warmup_runs=10,
                              label=f"  Inference")
        throughput_infer = batch_size / (r_infer['mean_ms'] / 1000)
        r_infer['throughput_samples_per_sec'] = float(throughput_infer)
        tokens_per_sec = (batch_size * seq_len) / (r_infer['mean_ms'] / 1000)
        r_infer['tokens_per_sec'] = float(tokens_per_sec)
        print(f"    → {throughput_infer:.1f} samples/sec, {tokens_per_sec:.0f} tokens/sec")

        # Training
        model.train()
        def train_fn():
            optimizer.zero_grad()
            output = model(x)
            loss = criterion(output, y)
            loss.backward()
            optimizer.step()

        r_train = benchmark_fn(train_fn, num_runs=30, warmup_runs=10,
                              label=f"  Training step")
        throughput_train = batch_size / (r_train['mean_ms'] / 1000)
        r_train['throughput_samples_per_sec'] = float(throughput_train)
        tokens_per_sec_train = (batch_size * seq_len) / (r_train['mean_ms'] / 1000)
        r_train['tokens_per_sec'] = float(tokens_per_sec_train)
        print(f"    → {throughput_train:.1f} samples/sec, {tokens_per_sec_train:.0f} tokens/sec")

        mem = get_gpu_memory()
        print(f"    Peak memory: {mem['max_allocated']:.1f} MB")

        results['benchmarks']['transformer_training'][label] = {
            'inference': r_infer,
            'training': r_train,
            'peak_memory_mb': mem['max_allocated'],
            'param_count': param_count,
            'config': {
                'd_model': d_model, 'nhead': nhead, 'layers': num_layers,
                'ff': dim_ff, 'seq_len': seq_len, 'batch': batch_size
            }
        }

        del model, optimizer, criterion, x, y
        torch.cuda.empty_cache()
        gc.collect()

    except RuntimeError as e:
        print(f"  SKIPPED: {e}")
        results['benchmarks']['transformer_training'][label] = {'error': str(e)}

print("\n✅ Transformer training benchmark complete!")

BENCHMARK 6: TRANSFORMER MODEL TRAINING

--- Small-Transformer ---
  Config: d_model=256, heads=4, layers=4, ff=1024, seq=128, batch=64
  Parameters: 10,872,322
    Inference: 12.980 ± 0.107 ms (min=12.608, median=13.007)
    → 4930.5 samples/sec, 631105 tokens/sec
    Training step: 64.582 ± 0.084 ms (min=64.440, median=64.576)
    → 991.0 samples/sec, 126846 tokens/sec
    Peak memory: 999.0 MB

--- BERT-base-like ---
  Config: d_model=512, heads=8, layers=6, ff=2048, seq=128, batch=32
  Parameters: 34,340,866
    Inference: 32.176 ± 0.098 ms (min=31.986, median=32.174)
    → 994.5 samples/sec, 127300 tokens/sec
    Training step: 122.852 ± 0.143 ms (min=122.531, median=122.823)
    → 260.5 samples/sec, 33341 tokens/sec
    Peak memory: 1655.4 MB

--- BERT-long-seq ---
  Config: d_model=512, heads=8, layers=6, ff=2048, seq=256, batch=16
  Parameters: 34,406,402
    Inference: 35.173 ± 0.121 ms (min=34.874, median=35.193)
    → 454.9 samples/sec, 116451 tokens/sec
    Training step: 1

## Memory, Data Transfer & Save

In [8]:
# ============================================================
# PYTORCH BENCHMARK - Cell 8: Memory, Data Transfer & Save
# ============================================================
print("=" * 70)
print("BENCHMARK 7: MEMORY & DATA TRANSFER")
print("=" * 70)

results['benchmarks']['memory'] = {}

# Host-to-Device and Device-to-Host transfers
print("\n--- Memory Transfer (CPU ↔ GPU) ---")
transfer_sizes_mb = [1, 10, 50, 100, 500, 1000]

results['benchmarks']['memory']['h2d'] = {}
results['benchmarks']['memory']['d2h'] = {}

for size_mb in transfer_sizes_mb:
    numel = size_mb * 1024 * 1024 // 4  # float32

    try:
        # Host to Device
        cpu_tensor = torch.randn(numel, dtype=torch.float32)

        def h2d_fn():
            gpu_tensor = cpu_tensor.to(device)
            torch.cuda.synchronize()
            return gpu_tensor

        r_h2d = benchmark_fn(h2d_fn, num_runs=50, warmup_runs=10,
                            label=f"  H2D {size_mb} MB")
        bandwidth_h2d = size_mb / (r_h2d['mean_ms'] / 1000) / 1000  # GB/s
        r_h2d['bandwidth_gbs'] = float(bandwidth_h2d)
        print(f"    → {bandwidth_h2d:.2f} GB/s")
        results['benchmarks']['memory']['h2d'][f'{size_mb}MB'] = r_h2d

        # Device to Host
        gpu_tensor = torch.randn(numel, dtype=torch.float32, device=device)

        def d2h_fn():
            cpu_t = gpu_tensor.cpu()
            torch.cuda.synchronize()
            return cpu_t

        r_d2h = benchmark_fn(d2h_fn, num_runs=50, warmup_runs=10,
                            label=f"  D2H {size_mb} MB")
        bandwidth_d2h = size_mb / (r_d2h['mean_ms'] / 1000) / 1000
        r_d2h['bandwidth_gbs'] = float(bandwidth_d2h)
        print(f"    → {bandwidth_d2h:.2f} GB/s")
        results['benchmarks']['memory']['d2h'][f'{size_mb}MB'] = r_d2h

        del cpu_tensor, gpu_tensor
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"  {size_mb} MB: ERROR - {e}")

# GPU memory allocation/deallocation benchmark
print("\n--- GPU Memory Allocation ---")
results['benchmarks']['memory']['allocation'] = {}
for size_mb in [10, 100, 500, 1000]:
    numel = size_mb * 1024 * 1024 // 4

    try:
        def alloc_fn():
            t = torch.empty(numel, dtype=torch.float32, device=device)
            del t
            torch.cuda.synchronize()

        r = benchmark_fn(alloc_fn, num_runs=100, warmup_runs=20,
                        label=f"  Alloc {size_mb} MB")
        results['benchmarks']['memory']['allocation'][f'{size_mb}MB'] = r
    except:
        pass

# --- Save Results ---
print("\n" + "=" * 70)
print("SAVING RESULTS")
print("=" * 70)

# Add timestamp
import datetime
results['timestamp'] = datetime.datetime.now().isoformat()

# Save to JSON
output_file = 'benchmark_pytorch_results.json'
with open(output_file, 'w') as f:
    json.dump(results, f, indent=2)

print(f"\n✅ Results saved to: {output_file}")
print(f"\nTotal benchmarks completed: {sum(len(v) if isinstance(v, dict) else 1 for v in results['benchmarks'].values())}")

# Print summary
print("\n" + "=" * 70)
print("SUMMARY")
print("=" * 70)

if 'matmul' in results['benchmarks']:
    if 'float32' in results['benchmarks']['matmul']:
        mm = results['benchmarks']['matmul']['float32']
        if '4096' in mm and 'tflops' in mm['4096']:
            print(f"MatMul 4096x4096 FP32: {mm['4096']['tflops']:.2f} TFLOPS")
    if 'float16' in results['benchmarks']['matmul']:
        mm = results['benchmarks']['matmul']['float16']
        if '4096' in mm and 'tflops' in mm['4096']:
            print(f"MatMul 4096x4096 FP16: {mm['4096']['tflops']:.2f} TFLOPS")

if 'cnn_training' in results['benchmarks']:
    if 'batch_32' in results['benchmarks']['cnn_training']:
        ct = results['benchmarks']['cnn_training']['batch_32']
        if 'training' in ct:
            print(f"ResNet Training (batch=32): {ct['training'].get('throughput_imgs_per_sec', 'N/A'):.1f} img/s")

if 'transformer_training' in results['benchmarks']:
    if 'BERT-base-like' in results['benchmarks']['transformer_training']:
        tt = results['benchmarks']['transformer_training']['BERT-base-like']
        if 'training' in tt:
            print(f"BERT Training: {tt['training'].get('tokens_per_sec', 'N/A'):.0f} tokens/sec")

print("\n🏁 PyTorch benchmark complete!")

BENCHMARK 7: MEMORY & DATA TRANSFER

--- Memory Transfer (CPU ↔ GPU) ---
    H2D 1 MB: 0.186 ± 0.029 ms (min=0.139, median=0.185)
    → 5.37 GB/s
    D2H 1 MB: 0.155 ± 0.051 ms (min=0.136, median=0.141)
    → 6.45 GB/s
    H2D 10 MB: 0.786 ± 0.036 ms (min=0.763, median=0.768)
    → 12.72 GB/s
    D2H 10 MB: 0.788 ± 0.028 ms (min=0.774, median=0.777)
    → 12.70 GB/s
    H2D 50 MB: 3.767 ± 0.054 ms (min=3.701, median=3.782)
    → 13.27 GB/s
    D2H 50 MB: 18.687 ± 0.239 ms (min=18.204, median=18.680)
    → 2.68 GB/s
    H2D 100 MB: 7.501 ± 0.046 ms (min=7.394, median=7.501)
    → 13.33 GB/s
    D2H 100 MB: 34.577 ± 1.042 ms (min=33.728, median=34.283)
    → 2.89 GB/s
    H2D 500 MB: 37.324 ± 0.090 ms (min=37.237, median=37.310)
    → 13.40 GB/s
    D2H 500 MB: 167.144 ± 4.352 ms (min=163.705, median=165.904)
    → 2.99 GB/s
    H2D 1000 MB: 75.273 ± 0.690 ms (min=74.495, median=75.050)
    → 13.28 GB/s
    D2H 1000 MB: 348.839 ± 5.772 ms (min=339.970, median=347.415)
    → 2.87 GB/s

--